# Term SOFR Approximation Calculator

Approximates forward-looking Term SOFR rates **(1M, 3M, 6M, 12M)** by bootstrapping implied rates from CME SR1 (1-month) and SR3 (3-month) SOFR futures prices sourced from Yahoo Finance via `yfinance`.

---

> **DISCLAIMER:** This produces an **approximation** of Term SOFR — not the official CME fixing.  
> CME uses intraday VWAP; this notebook uses end-of-day settlement prices.  
> Typical deviation: 0.5–3 basis points.  
> Suitable for internal modelling, budgeting, and underwriting.  
> **NOT suitable as a contractual reference rate.**

## 1 — Install Dependencies

`domojupyter` is pre-installed in Domo's Jupyter environment. Only external packages need installing.

In [ ]:
!pip install yfinance pandas numpy requests --quiet

## 2 — Imports

In [ ]:
import calendar
from datetime import date, timedelta
from typing import Optional

import numpy as np
import pandas as pd
import requests
import yfinance as yf
import domojupyter as domo

## 3 — Configuration

Edit the variables in this cell before running.

| Variable | Description |
|---|---|
| `OUTPUT_ALIAS` | Domo output alias — both the futures-implied term SOFR and NY Fed official rates write here |
| `UPDATE_METHOD` | `'REPLACE'` overwrites each run; `'APPEND'` adds rows (the NY Fed section always appends after) |
| `HISTORY_DAYS` | `None` = today only; integer (e.g. `60`) = historical backfill for futures section |

> **Setting the alias:** In Domo, open this notebook → **Data** tab → add an output dataset and name it (e.g. `sofr_rates`). Use that exact name as `OUTPUT_ALIAS` below.

In [ ]:
# ── USER CONFIG ───────────────────────────────────────────────────────────────
OUTPUT_ALIAS:  str           = "term_sofr_rates"  # must match the alias in Domo's Data tab
UPDATE_METHOD: str           = "REPLACE"     # "REPLACE" | "APPEND"
HISTORY_DAYS:  Optional[int] = None          # None = today only; e.g. 60 for backfill
# ─────────────────────────────────────────────────────────────────────────────

IS_HISTORICAL = HISTORY_DAYS is not None

if IS_HISTORICAL and HISTORY_DAYS > 90:
    print(f"WARNING: HISTORY_DAYS={HISTORY_DAYS} exceeds the recommended limit of 90.")
    print("Yahoo Finance does not serve historical prices for expired CME futures.")
    print("Contracts older than ~60-90 days will be missing, producing incomplete results.")
    print("For deep history, a paid data source (Quandl, Barchart, CME DataMine) is required.\n")

print(f"Output alias  : {OUTPUT_ALIAS}")
print(f"Update method : {UPDATE_METHOD}")
print(f"Mode          : {'Historical (' + str(HISTORY_DAYS) + ' days)' if IS_HISTORICAL else 'Current (today only)'}")

## 4 — Contract Calendar Helpers

In [ ]:
# CME month codes: F=Jan G=Feb H=Mar J=Apr K=May M=Jun N=Jul Q=Aug U=Sep V=Oct X=Nov Z=Dec
MONTH_CODES = {
    1:"F", 2:"G", 3:"H", 4:"J", 5:"K", 6:"M",
    7:"N", 8:"Q", 9:"U", 10:"V", 11:"X", 12:"Z"
}

SR3_MONTHS = {3, 6, 9, 12}  # SR3 quarterly expiries: Mar/Jun/Sep/Dec


def get_sr1_tickers(n: int = 14, ref_date: Optional[date] = None) -> list[dict]:
    """Next `n` SR1 (1-month SOFR) futures tickers from ref_date."""
    ref = ref_date or date.today()
    tickers = []
    year, month = ref.year, ref.month
    for _ in range(n):
        code = MONTH_CODES[month]
        yy   = str(year)[-2:]
        tickers.append({
            "ticker":      f"SR1{code}{yy}.CME",
            "year":        year,
            "month":       month,
            "expiry_date": date(year, month, 1),
            "type":        "SR1",
        })
        month += 1
        if month > 12:
            month = 1
            year += 1
    return tickers


def get_sr3_tickers(n: int = 6, ref_date: Optional[date] = None) -> list[dict]:
    """Next `n` SR3 (3-month SOFR) quarterly futures tickers from ref_date."""
    ref = ref_date or date.today()
    tickers = []
    year, month = ref.year, ref.month
    while month not in SR3_MONTHS:
        month += 1
        if month > 12:
            month = 1
            year += 1
    for _ in range(n):
        code = MONTH_CODES[month]
        yy   = str(year)[-2:]
        tickers.append({
            "ticker":      f"SR3{code}{yy}.CME",
            "year":        year,
            "month":       month,
            "expiry_date": date(year, month, 1),
            "type":        "SR3",
        })
        month += 3
        if month > 12:
            month -= 12
            year += 1
    return tickers

## 5 — Fetch Futures Prices

In [ ]:
def fetch_prices(contracts: list[dict], period: str = "5d") -> dict[str, float]:
    """Fetch latest closing prices for contracts from Yahoo Finance."""
    tickers_list = [c["ticker"] for c in contracts]
    prices = {}
    print(f"  Fetching {len(tickers_list)} contracts from Yahoo Finance...")
    try:
        raw = yf.download(tickers_list, period=period, progress=False, auto_adjust=True)
        if raw.empty:
            print("  WARNING: No data returned.")
            return prices
        close = raw["Close"] if "Close" in raw.columns else raw
        for ticker in tickers_list:
            try:
                series = close[ticker].dropna() if ticker in close.columns else close.dropna()
                if not series.empty:
                    prices[ticker] = float(series.iloc[-1])
            except Exception:
                pass
    except Exception as e:
        print(f"  ERROR fetching prices: {e}")
    valid = {k: v for k, v in prices.items() if v > 50}
    print(f"  Retrieved {len(valid)}/{len(tickers_list)} valid contract prices.")
    return valid


def fetch_prices_historical(
    contracts: list[dict],
    start: str,
    end: str,
) -> pd.DataFrame:
    """Fetch daily closing prices for all contracts over a date range."""
    tickers_list = [c["ticker"] for c in contracts]
    print(f"  Fetching historical prices for {len(tickers_list)} contracts ({start} to {end})...")
    try:
        raw = yf.download(tickers_list, start=start, end=end, progress=False, auto_adjust=True)
        if raw.empty:
            return pd.DataFrame()
        close = raw["Close"] if "Close" in raw.columns else raw
        close = close.loc[:, (close > 50).any()]
        return close
    except Exception as e:
        print(f"  ERROR: {e}")
        return pd.DataFrame()

## 6 — Overnight SOFR (NY Fed)

In [ ]:
def fetch_overnight_sofr() -> Optional[float]:
    """Fetch latest overnight SOFR from the NY Fed public API."""
    try:
        url  = "https://markets.newyorkfed.org/api/rates/sofr/last/1.json"
        r    = requests.get(url, timeout=10)
        r.raise_for_status()
        rate = r.json()["refRates"][0]["percentRate"]
        print(f"  Overnight SOFR (NY Fed): {rate:.4f}%")
        return float(rate)
    except Exception as e:
        print(f"  WARNING: Could not fetch overnight SOFR — {e}")
        return None

## 7 — Core Calculation

In [ ]:
def futures_price_to_rate(price: float) -> float:
    return 100.0 - price


def days_in_month(year: int, month: int) -> int:
    return calendar.monthrange(year, month)[1]


def build_implied_curve(
    sr1_contracts: list[dict],
    sr3_contracts: list[dict],
    prices: dict[str, float],
    overnight_sofr: Optional[float],
    ref_date: date,
    min_forward_days: int = 365,
) -> list[dict]:
    """Bootstrap a daily forward rate curve from SR1 futures prices.

    If available contract data covers fewer than min_forward_days, the last
    known rate is carried forward (flat extrapolation).  This prevents longer
    tenors from returning None when far-out contracts lack historical prices.
    """
    curve = []
    for contract in sr1_contracts:
        ticker = contract["ticker"]
        if ticker not in prices:
            continue
        rate   = futures_price_to_rate(prices[ticker])
        year, month = contract["year"], contract["month"]
        n_days = days_in_month(year, month)
        d   = date(year, month, 1)
        end = date(year, month, n_days)
        while d <= end:
            if d >= ref_date:
                curve.append({"date": d.isoformat(), "implied_rate": rate, "source": ticker})
            d += timedelta(days=1)

    curve.sort(key=lambda x: x["date"])

    # Carry the last rate forward so all four tenors can always be computed
    if curve and min_forward_days:
        last      = curve[-1]
        last_date = date.fromisoformat(last["date"])
        target    = ref_date + timedelta(days=min_forward_days)
        if last_date < target:
            d = last_date + timedelta(days=1)
            while d <= target:
                curve.append({
                    "date":         d.isoformat(),
                    "implied_rate": last["implied_rate"],
                    "source":       last["source"] + " [flat ext.]",
                })
                d += timedelta(days=1)

    return curve


def compound_rate(curve_slice: list[dict], tenor_days: int) -> Optional[float]:
    """Compound daily implied rates over tenor_days using Act/360 SOFR convention."""
    if len(curve_slice) < tenor_days:
        return None
    compounded = 1.0
    for point in curve_slice[:tenor_days]:
        compounded *= (1 + (point["implied_rate"] / 100.0) * (1 / 360))
    return round((compounded - 1) * (360 / tenor_days) * 100, 5)


TENORS = {
    "1-Month Term SOFR":  30,
    "3-Month Term SOFR":  90,
    "6-Month Term SOFR":  180,
    "12-Month Term SOFR": 360,
}


def calculate_term_sofr(curve: list[dict], ref_date: date) -> dict[str, Optional[float]]:
    forward = [p for p in curve if p["date"] >= ref_date.isoformat()]
    return {label: compound_rate(forward, days) for label, days in TENORS.items()}

## 8 — Output Helpers

In [ ]:
def results_to_dataframe(results: dict, ref_date: date, overnight: Optional[float]) -> pd.DataFrame:
    rows = []
    if overnight:
        rows.append({
            "Date":       ref_date.isoformat(),
            "Tenor":      "Overnight",
            "Rate (%)":   round(overnight, 5),
            "Type":       "Overnight SOFR",
            "Source":     "NY Fed API",
            "Note":       "Official fixing",
            "Volume (B)": None,
        })
    for label, rate in results.items():
        rows.append({
            "Date":       ref_date.isoformat(),
            "Tenor":      label.replace(" Term SOFR", "").replace("-", "").strip(),
            "Rate (%)":   round(rate, 5) if rate is not None else None,
            "Type":       "Term SOFR (Approximated)",
            "Source":     "CME SR1 Futures via Yahoo Finance",
            "Note":       "Bootstrapped from EOD settlement — not official CME fixing",
            "Volume (B)": None,
        })
    return pd.DataFrame(rows)


def write_to_domo(df: pd.DataFrame, alias: str, update_method: str = "REPLACE") -> None:
    if df.empty:
        print("No data to write — skipping Domo upload.")
        return
    domo.write_dataframe(df, alias, update_method=update_method)
    print(f"Written {len(df)} rows to Domo output '{alias}' ({update_method}).")

## 9 — Run: Current Mode

Calculates implied Term SOFR rates as of **today** and writes them to the configured Domo dataset.  
Skipped automatically when `HISTORY_DAYS` is set.

> For a **daily scheduled run**, set `UPDATE_METHOD = "APPEND"` so each execution adds a new row rather than overwriting the full dataset.

In [ ]:
if not IS_HISTORICAL:
    ref_date = date.today()
    print(f"\nCurrent mode: calculating implied Term SOFR as of {ref_date}\n")

    sr1_contracts = get_sr1_tickers(n=14, ref_date=ref_date)
    sr3_contracts = get_sr3_tickers(n=6,  ref_date=ref_date)

    prices    = fetch_prices(sr1_contracts + sr3_contracts)
    overnight = fetch_overnight_sofr()

    if not prices:
        print("ERROR: No futures prices retrieved. Check internet connection.")
    else:
        curve      = build_implied_curve(sr1_contracts, sr3_contracts, prices, overnight, ref_date)
        results    = calculate_term_sofr(curve, ref_date)
        df_current = results_to_dataframe(results, ref_date, overnight)

        display(df_current)
        write_to_domo(df_current, OUTPUT_ALIAS, UPDATE_METHOD)
else:
    print("Historical mode selected — skipping current-mode cell.")

## 10 — Run: Historical Mode

Calculates implied Term SOFR rates over the last `HISTORY_DAYS` calendar days and writes the full history to the configured Domo dataset. Skipped when `HISTORY_DAYS` is `None`.

> **Yahoo Finance limitation:** Expired CME futures are removed from Yahoo Finance after settlement. This caps the reliable historical lookback at roughly **60–90 days**. Setting `HISTORY_DAYS` beyond that will produce warnings for missing contracts and sparse/incomplete output for older dates.
>
> For a deeper backfill, a paid data source is required (e.g. Quandl/NASDAQ Data Link, Barchart, or CME DataMine).
>
> **Recommended workflow:** Run the historical backfill once with `HISTORY_DAYS = 60` and `UPDATE_METHOD = "REPLACE"`, then switch to current mode with `UPDATE_METHOD = "APPEND"` on a daily schedule to build up history going forward.

In [ ]:
if IS_HISTORICAL:
    end_date   = date.today()
    start_date = end_date - timedelta(days=HISTORY_DAYS)
    print(f"\nHistorical mode: {start_date} to {end_date} ({HISTORY_DAYS} days)\n")

    # Start contract generation from start_date so expired contracts are included.
    # n must cover the lookback period plus a 12-month forward window for the longest tenor.
    months_span   = HISTORY_DAYS // 30 + 14
    quarters_span = HISTORY_DAYS // 90 + 6
    sr1_contracts = get_sr1_tickers(n=months_span,   ref_date=start_date)
    sr3_contracts = get_sr3_tickers(n=quarters_span, ref_date=start_date)
    all_contracts = sr1_contracts + sr3_contracts

    price_df = fetch_prices_historical(
        all_contracts,
        start=start_date.isoformat(),
        end=(end_date + timedelta(days=1)).isoformat(),
    )

    if price_df.empty:
        print("ERROR: No historical price data retrieved.")
    else:
        history_rows = []
        for ts, row in price_df.iterrows():
            ref = ts.date() if hasattr(ts, "date") else date.fromisoformat(str(ts)[:10])
            day_prices = {col: float(val) for col, val in row.items()
                          if pd.notna(val) and float(val) > 50}
            if not day_prices:
                continue
            curve = build_implied_curve(sr1_contracts, sr3_contracts, day_prices, None, ref)
            if not curve:
                continue
            for label, rate in calculate_term_sofr(curve, ref).items():
                if rate is not None:
                    history_rows.append({
                        "Date":     ref.isoformat(),
                        "Tenor":    label.replace(" Term SOFR", "").replace("-", "").strip(),
                        "Rate (%)": round(rate, 5),
                        "Type":     "Term SOFR (Approximated)",
                        "Source":   "CME SR1 Futures via Yahoo Finance",
                        "Note":     "Bootstrapped from EOD settlement — not official CME fixing",
                    })

        df_history = pd.DataFrame(history_rows)
        if not df_history.empty:
            n_obs  = len(df_history)
            n_days = df_history["Date"].nunique()
            print(f"Calculated {n_obs} rate observations across {n_days} trading days.\n")

            # Pivot for readability: rows = dates, columns = tenors
            pivot = (
                df_history
                .pivot(index="Date", columns="Tenor", values="Rate (%)")
                .rename_axis(None, axis=1)
                .reset_index()
            )
            display(pivot)

            write_to_domo(df_history, OUTPUT_ALIAS, UPDATE_METHOD)
        else:
            print("No rates could be calculated from available data.")
else:
    print("Current mode selected — skipping historical-mode cell.")

## 11 — Run: NY Fed Official SOFR Rates (Prior Business Day)

Fetches the prior business day's official published SOFR rates and **appends** them to the same Domo dataset as the futures-implied rates.

| Data | Tenor | Source |
|---|---|---|
| Overnight SOFR + volume | `Overnight` | NY Fed secured rates API |
| 30-Day Average SOFR | `30DayAvg` | NY Fed via FRED public CSV |
| 90-Day Average SOFR | `90DayAvg` | NY Fed via FRED public CSV |
| 180-Day Average SOFR | `180DayAvg` | NY Fed via FRED public CSV |

> These are **backward-looking** compounded averages, distinct from the forward-looking futures-implied term rates. Use the `Type` column in Domo to filter between them.

In [ ]:
NYFED_TYPE_MAP = {
    "SOFR":          ("Overnight", "Overnight SOFR"),
    "SOFR30DAYAVG":  ("30DayAvg",  "30-Day Average SOFR"),
    "SOFR90DAYAVG":  ("90DayAvg",  "90-Day Average SOFR"),
    "SOFR180DAYAVG": ("180DayAvg", "180-Day Average SOFR"),
}

FRED_SERIES = {
    "SOFR30DAYAVG":  "30DayAvg",
    "SOFR90DAYAVG":  "90DayAvg",
    "SOFR180DAYAVG": "180DayAvg",
}


def prior_business_day() -> date:
    d = date.today() - timedelta(days=1)
    while d.weekday() >= 5:
        d -= timedelta(days=1)
    return d


def _nyfed_search(target: str, query_type: str) -> list[dict]:
    """Call the NY Fed secured SOFR search endpoint for a given query type."""
    url = (
        "https://markets.newyorkfed.org/api/rates/secured/sofr/search.json"
        f"?startDate={target}&endDate={target}&type={query_type}"
    )
    try:
        r = requests.get(url, timeout=30)
        r.raise_for_status()
        return r.json().get("refRates", [])
    except Exception as e:
        print(f"  WARNING (NY Fed {query_type}): {e}")
        return []


def _fetch_fred_average(series_id: str, target: str) -> Optional[float]:
    """Fetch the most recent value on or before target from a FRED public CSV."""
    url = f"https://fred.stlouisfed.org/graph/fredgraph.csv?id={series_id}"
    try:
        r = requests.get(url, timeout=30)
        r.raise_for_status()
        for line in reversed(r.text.strip().splitlines()[1:]):  # newest first
            parts = line.split(",")
            if len(parts) != 2:
                continue
            d, val = parts[0].strip(), parts[1].strip()
            if d > target or not val or val == ".":
                continue
            return round(float(val), 5)
    except Exception as e:
        print(f"  WARNING ({series_id}): {e}")
    return None


def fetch_nyfed_sofr_prior_day() -> pd.DataFrame:
    """Overnight SOFR + volume from NY Fed; 30/90/180-day averages from FRED."""
    target = prior_business_day().isoformat()
    rows   = []

    # ── Rates ─────────────────────────────────────────────────────────────────
    rate_items = _nyfed_search(target, "rate")

    # ── Volumes (separate call) — build lookup: api_type → volume (B) ─────────
    volume_lookup: dict[str, Optional[float]] = {}
    for item in _nyfed_search(target, "volume"):
        api_type = item.get("type", "")
        vol      = item.get("volume")
        if api_type and vol is not None:
            volume_lookup[api_type] = round(float(vol), 2)

    # ── Combine rate + volume for overnight ───────────────────────────────────
    for item in rate_items:
        api_type = item.get("type", "")
        if api_type not in NYFED_TYPE_MAP:
            continue
        tenor, type_label = NYFED_TYPE_MAP[api_type]
        rows.append({
            "Date":       item["effectiveDate"],
            "Tenor":      tenor,
            "Rate (%)":   round(float(item["percentRate"]), 5),
            "Type":       type_label,
            "Source":     "NY Fed API",
            "Note":       "Official published rate",
            "Volume (B)": volume_lookup.get(api_type),
        })

    # ── 30 / 90 / 180-day averages from FRED ─────────────────────────────────
    for series_id, tenor in FRED_SERIES.items():
        rate = _fetch_fred_average(series_id, target)
        if rate is not None:
            rows.append({
                "Date":       target,
                "Tenor":      tenor,
                "Rate (%)":   rate,
                "Type":       NYFED_TYPE_MAP[series_id][1],
                "Source":     "NY Fed via FRED",
                "Note":       "Official published rate",
                "Volume (B)": None,
            })

    if not rows:
        print(f"  WARNING: No data for {target} — may be a holiday.")
        return pd.DataFrame()

    return pd.DataFrame(rows).sort_values("Tenor").reset_index(drop=True)


# ── Run ───────────────────────────────────────────────────────────────────────
target_date = prior_business_day().isoformat()
print(f"\nNY Fed SOFR rates for {target_date} (prior business day)\n")

df_nyfed = fetch_nyfed_sofr_prior_day()

if not df_nyfed.empty:
    display(df_nyfed[["Tenor", "Rate (%)", "Volume (B)", "Type"]])
    write_to_domo(df_nyfed, OUTPUT_ALIAS, "APPEND")
else:
    print(f"No data written — {target_date} may be a holiday with no NY Fed publication.")